In [0]:
df = spark.read.csv(
    "/Volumes/workspace/default/bronze_volume/ecommerce_sample.csv",
    header=True,
    inferSchema=True
)

df.show(5)

In [0]:
import re

def clean_column(col):
    col = col.strip()  # hapus spasi depan belakang
    col = col.lower()  # jadi huruf kecil
    col = re.sub(r'[ ,;{}()\n\t=]+', '_', col)  # ganti karakter aneh jadi _
    return col

df = df.toDF(*[clean_column(c) for c in df.columns])

In [0]:
df.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.default.bronze_ecommerce")
    

In [0]:
from pyspark.sql.functions import col

df_bronze = spark.table("workspace.default.bronze_ecommerce")

df_silver = df_bronze.dropDuplicates()

df_silver = df_silver.fillna({
    "qty_ordered": 0,
    "grand_total": 0
})

df_silver = df_silver.filter(col("qty_ordered") > 0)

In [0]:
from pyspark.sql.functions import col

df_silver = df_silver.withColumn(
    "net_revenue",
    col("grand_total") - col("discount_amount")
)

In [0]:
df_silver.write.format("delta") \
    .saveAsTable("workspace.default.silver_ecommerce")

In [0]:
from pyspark.sql.functions import sum, count, avg

df_gold_kpi = df_silver.groupBy("working_date").agg(
    sum("net_revenue").alias("total_revenue"),
    count("increment_id").alias("total_orders"),
    avg("net_revenue").alias("avg_order_value"),
    sum("qty_ordered").alias("total_quantity")
)

In [0]:
df_gold_kpi.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("workspace.default.gold_kpi")

In [0]:
df_gold = spark.table("workspace.default.gold_kpi")
display(df_gold)

In [0]:
df_silver = df_silver.filter(col("qty_ordered") > 0)


In [0]:
sum("value")
avg("value")

In [0]:
from pyspark.sql.functions import sum, count, avg

df_gold = df_silver.groupBy("working_date").agg(
    sum("grand_total").alias("total_revenue"),
    count("increment_id").alias("total_orders"),
    avg("grand_total").alias("avg_order_value")
)

In [0]:
df_bronze = spark.table("workspace.default.bronze_ecommerce")

In [0]:
df_silver.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.default.silver_ecommerce")
    

In [0]:
spark.table("workspace.default.silver_ecommerce").show(5)

In [0]:
df_valid = df_silver.filter(df_silver.status == "complete")

In [0]:
df_silver = df_silver.drop(
    "unnamed:_21"
)

In [0]:
df_silver = df_silver.filter("status = 'complete'")

In [0]:
df_silver = spark.table("workspace.default.silver_ecommerce")

In [0]:
from pyspark.sql.functions import col

df_silver = df_silver.withColumn(
    "net_revenue",
    col("grand_total") - col("discount_amount")
)

In [0]:
spark.table("workspace.default.gold_kpi").show(5)

In [0]:
from pyspark.sql.functions import round

df_gold_kpi = df_gold_kpi.select(
    "working_date",
    round("total_revenue", 2).alias("total_revenue"),
    "total_orders",
    round("avg_order_value", 2).alias("avg_order_value"),
    "total_quantity"
)

In [0]:
df_gold_kpi.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("workspace.default.gold_kpi")

In [0]:
%pip install openpyxl


In [0]:
dbutils.library.restartPython()

In [0]:
df_gold = spark.table("workspace.default.gold_kpi")

pdf = df_gold.toPandas()

pdf.to_excel("/tmp/gold_kpi.xlsx", index=False)

In [0]:
df_bronze = spark.table("workspace.default.bronze_ecommerce")

In [0]:
df_gold = spark.table("workspace.default.gold_kpi")

display(df_gold)

In [0]:
df = spark.table("workspace.default.gold_kpi")

df = df.dropna()

w = Window.orderBy("working_date")

df_feature = df.withColumn(
    "day_index", row_number().over(w) - 1
).withColumn(
    "month", month("working_date")
).withColumn(
    "quarter", quarter("working_date")
).withColumn(
    "day_of_week", dayofweek("working_date")
).withColumn(
    "day_of_month", dayofmonth("working_date")
).withColumn(
    "week_of_year", weekofyear("working_date")
).withColumn(
    "is_weekend", when(col("day_of_week").isin(1, 7), 1).otherwise(0)
).withColumn(
    "is_payday", when(col("day_of_month") >= 25, 1).otherwise(0)
).withColumn(
    "is_high_season", when(col("month").isin(11, 12), 1).otherwise(0)
).withColumn(
    "is_low_season", when(col("month").isin(2, 6), 1).otherwise(0)
).withColumn(
    "is_event_day",
    when(
        ((col("month") == 11) & (col("day_of_month") == 11)) |
        ((col("month") == 12) & (col("day_of_month") == 12)),
        1
    ).otherwise(0)
)

w_lag = Window.orderBy("working_date")

df_feature = df_feature.withColumn("lag_orders_1", lag("total_orders", 1).over(w_lag)) \
    .withColumn("lag_orders_7", lag("total_orders", 7).over(w_lag)) \
    .withColumn("lag_revenue_1", lag("total_revenue", 1).over(w_lag)) \
    .withColumn("lag_revenue_7", lag("total_revenue", 7).over(w_lag))

w_roll_7 = Window.orderBy("working_date").rowsBetween(-7, -1)
w_roll_30 = Window.orderBy("working_date").rowsBetween(-30, -1)

df_feature = df_feature.withColumn("rolling_orders_7", avg("total_orders").over(w_roll_7)) \
    .withColumn("rolling_orders_30", avg("total_orders").over(w_roll_30)) \
    .withColumn("rolling_revenue_7", avg("total_revenue").over(w_roll_7)) \
    .withColumn("rolling_revenue_30", avg("total_revenue").over(w_roll_30))

df_feature = df_feature.fillna(0)

display(df_feature)

In [0]:
avg_revenue = df_feature.select(avg("total_revenue")).collect()[0][0]

df_classification = df_feature.withColumn(
    "sales_category",
    when(col("total_revenue") >= avg_revenue, "High Sales").otherwise("Low Sales")
)

classification_assembler = VectorAssembler(
    inputCols=[
        "total_orders",
        "total_quantity",
        "month",
        "quarter",
        "day_of_week",
        "is_weekend",
        "is_payday",
        "is_high_season",
        "is_low_season",
        "is_event_day"
    ],
    outputCol="features"
)

label_indexer = StringIndexer(
    inputCol="sales_category",
    outputCol="label"
)

dt = DecisionTreeClassifier(
    featuresCol="features",
    labelCol="label",
    predictionCol="prediction",
    maxDepth=5
)

pipeline = Pipeline(
    stages=[
        classification_assembler,
        label_indexer,
        dt
    ]
)

train_cls, test_cls = df_classification.randomSplit([0.8, 0.2], seed=42)

classification_model = pipeline.fit(train_cls)

classification_predictions = classification_model.transform(test_cls)

accuracy = MulticlassClassificationEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="accuracy"
).evaluate(classification_predictions)

precision = MulticlassClassificationEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="weightedPrecision"
).evaluate(classification_predictions)

recall = MulticlassClassificationEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="weightedRecall"
).evaluate(classification_predictions)

print("Accuracy:", accuracy)
print("Precision:", precision)
print("Recall:", recall)

display(
    classification_predictions.select(
        "working_date",
        "total_revenue",
        "total_orders",
        "total_quantity",
        "sales_category",
        "prediction",
        "probability"
    )
)
(
    df_sales_classification_result.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("workspace.default.gold_sales_classification")
)

In [0]:
revenue_assembler = VectorAssembler(
    inputCols=[
        "day_index",
        "total_orders",
        "total_quantity",
        "month",
        "quarter",
        "day_of_week",
        "day_of_month",
        "week_of_year",
        "is_weekend",
        "is_payday",
        "is_high_season",
        "is_low_season",
        "is_event_day",
        "lag_revenue_1",
        "lag_revenue_7",
        "rolling_revenue_7",
        "rolling_revenue_30"
    ],
    outputCol="revenue_features"
)

df_revenue = revenue_assembler.transform(df_feature)

train_rev, test_rev = df_revenue.randomSplit([0.8, 0.2], seed=42)

revenue_gbt = GBTRegressor(
    featuresCol="revenue_features",
    labelCol="total_revenue",
    predictionCol="predicted_revenue",
    maxIter=100,
    maxDepth=5,
    stepSize=0.05,
    seed=42
)

revenue_model = revenue_gbt.fit(train_rev)

revenue_test_predictions = revenue_model.transform(test_rev).withColumn(
    "predicted_revenue",
    when(col("predicted_revenue") < 0, 0).otherwise(col("predicted_revenue"))
)

rmse_revenue = RegressionEvaluator(
    labelCol="total_revenue",
    predictionCol="predicted_revenue",
    metricName="rmse"
).evaluate(revenue_test_predictions)

mae_revenue = RegressionEvaluator(
    labelCol="total_revenue",
    predictionCol="predicted_revenue",
    metricName="mae"
).evaluate(revenue_test_predictions)

r2_revenue = RegressionEvaluator(
    labelCol="total_revenue",
    predictionCol="predicted_revenue",
    metricName="r2"
).evaluate(revenue_test_predictions)

print("Revenue RMSE:", rmse_revenue)
print("Revenue MAE:", mae_revenue)
print("Revenue R2:", r2_revenue)

In [0]:
last_info = df_feature.select(
    max("working_date").alias("last_date"),
    max("day_index").alias("last_day_index")
).collect()[0]

last_date = last_info["last_date"]
last_day_index = last_info["last_day_index"]

future_dates = spark.sql(f"""
SELECT explode(sequence(
    date_add(date('{last_date}'), 1),
    date_add(date('{last_date}'), 1825),
    interval 1 day
)) AS future_date
""")

w_future = Window.orderBy("future_date")

df_future = future_dates.withColumn(
    "day_index",
    lit(last_day_index) + row_number().over(w_future)
).withColumn(
    "month", month("future_date")
).withColumn(
    "quarter", quarter("future_date")
).withColumn(
    "day_of_week", dayofweek("future_date")
).withColumn(
    "day_of_month", dayofmonth("future_date")
).withColumn(
    "week_of_year", weekofyear("future_date")
).withColumn(
    "is_weekend", when(col("day_of_week").isin(1, 7), 1).otherwise(0)
).withColumn(
    "is_payday", when(col("day_of_month") >= 25, 1).otherwise(0)
).withColumn(
    "is_high_season", when(col("month").isin(11, 12), 1).otherwise(0)
).withColumn(
    "is_low_season", when(col("month").isin(2, 6), 1).otherwise(0)
).withColumn(
    "is_event_day",
    when(
        ((col("month") == 11) & (col("day_of_month") == 11)) |
        ((col("month") == 12) & (col("day_of_month") == 12)),
        1
    ).otherwise(0)
)

seasonal_avg = df_feature.groupBy("month", "day_of_week").agg(
    avg("total_orders").alias("avg_orders"),
    avg("total_quantity").alias("avg_quantity"),
    avg("rolling_revenue_7").alias("avg_rolling_revenue_7"),
    avg("rolling_revenue_30").alias("avg_rolling_revenue_30"),
    avg("lag_revenue_1").alias("avg_lag_revenue_1"),
    avg("lag_revenue_7").alias("avg_lag_revenue_7")
)

df_future = df_future.join(
    seasonal_avg,
    on=["month", "day_of_week"],
    how="left"
)

df_future = df_future.withColumn(
    "multiplier",
    when(col("is_event_day") == 1, 1.50)
    .when(col("is_high_season") == 1, 1.30)
    .when(col("is_low_season") == 1, 0.85)
    .otherwise(1.00)
)

df_future = df_future.withColumn(
    "total_orders",
    round(col("avg_orders") * col("multiplier"))
).withColumn(
    "total_quantity",
    round(col("avg_quantity") * col("multiplier"))
).withColumn(
    "lag_revenue_1",
    col("avg_lag_revenue_1")
).withColumn(
    "lag_revenue_7",
    col("avg_lag_revenue_7")
).withColumn(
    "rolling_revenue_7",
    col("avg_rolling_revenue_7")
).withColumn(
    "rolling_revenue_30",
    col("avg_rolling_revenue_30")
)

df_future = df_future.fillna(0)

future_revenue_feature = revenue_assembler.transform(df_future)

future_revenue_prediction = revenue_model.transform(future_revenue_feature).withColumn(
    "predicted_revenue",
    when(col("predicted_revenue") < 0, 0).otherwise(col("predicted_revenue"))
)

df_future_revenue_5_years = future_revenue_prediction.select(
    "future_date",
    "month",
    "quarter",
    "day_of_week",
    "is_weekend",
    "is_payday",
    "is_high_season",
    "is_low_season",
    "is_event_day",
    "total_orders",
    "total_quantity",
    "predicted_revenue"
)

(
    df_future_revenue_5_years.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("workspace.default.gold_enterprise_revenue_forecast_5_years")
)

display(df_future_revenue_5_years)

In [0]:
order_assembler = VectorAssembler(
    inputCols=[
        "day_index",
        "month",
        "quarter",
        "day_of_week",
        "day_of_month",
        "week_of_year",
        "is_weekend",
        "is_payday",
        "is_high_season",
        "is_low_season",
        "is_event_day",
        "lag_orders_1",
        "lag_orders_7",
        "rolling_orders_7",
        "rolling_orders_30"
    ],
    outputCol="order_features"
)

df_order = order_assembler.transform(df_feature)

train_order, test_order = df_order.randomSplit([0.8, 0.2], seed=42)

order_gbt = GBTRegressor(
    featuresCol="order_features",
    labelCol="total_orders",
    predictionCol="predicted_orders",
    maxIter=100,
    maxDepth=5,
    stepSize=0.05,
    seed=42
)

order_model = order_gbt.fit(train_order)

order_test_predictions = order_model.transform(test_order)

rmse_order = RegressionEvaluator(
    labelCol="total_orders",
    predictionCol="predicted_orders",
    metricName="rmse"
).evaluate(order_test_predictions)

mae_order = RegressionEvaluator(
    labelCol="total_orders",
    predictionCol="predicted_orders",
    metricName="mae"
).evaluate(order_test_predictions)

r2_order = RegressionEvaluator(
    labelCol="total_orders",
    predictionCol="predicted_orders",
    metricName="r2"
).evaluate(order_test_predictions)

print("Order RMSE:", rmse_order)
print("Order MAE:", mae_order)
print("Order R2:", r2_order)

future_order_avg = df_feature.groupBy("month", "day_of_week").agg(
    avg("lag_orders_1").alias("avg_lag_orders_1"),
    avg("lag_orders_7").alias("avg_lag_orders_7"),
    avg("rolling_orders_7").alias("avg_rolling_orders_7"),
    avg("rolling_orders_30").alias("avg_rolling_orders_30")
)

df_future_order = df_future.join(
    future_order_avg,
    on=["month", "day_of_week"],
    how="left"
).withColumn(
    "lag_orders_1", col("avg_lag_orders_1")
).withColumn(
    "lag_orders_7", col("avg_lag_orders_7")
).withColumn(
    "rolling_orders_7", col("avg_rolling_orders_7")
).withColumn(
    "rolling_orders_30", col("avg_rolling_orders_30")
).fillna(0)

future_order_feature = order_assembler.transform(df_future_order)

future_order_prediction = order_model.transform(future_order_feature)

future_order_prediction = future_order_prediction.withColumn(
    "predicted_orders",
    round(
        when(col("predicted_orders") < 0, 0).otherwise(col("predicted_orders")),
        0
    ).cast("int")
)

df_future_order_5_years = future_order_prediction.select(
    "future_date",
    "month",
    "quarter",
    "day_of_week",
    "is_weekend",
    "is_payday",
    "is_high_season",
    "is_low_season",
    "is_event_day",
    "predicted_orders"
)

(
    df_future_order_5_years.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("workspace.default.gold_enterprise_order_forecast_5_years")
)

display(df_future_order_5_years)

In [0]:
# =========================================================
# IMPORT LENGKAP DATA ENGINEERING & MACHINE LEARNING PROJECT
# =========================================================
# Project:
# 1. Sales Classification
# 2. Revenue Forecasting 5 Years
# 3. Order Volume Forecasting 5 Years
# 4. Enterprise Predictive Analytics
# =========================================================


# =========================================================
# PYSPARK SQL FUNCTIONS
# =========================================================

from pyspark.sql.functions import (

    # Basic Functions
    col,
    lit,
    when,

    # Aggregation
    max,
    min,
    avg,
    sum,
    count,

    # Numeric Functions
    round,
    abs,

    # Window Functions
    row_number,
    lag,

    # Date Functions
    month,
    quarter,
    weekofyear,
    dayofweek,
    dayofmonth,
    date_add,

    # Other Functions
    explode,
    sequence

)


# =========================================================
# WINDOW FUNCTION
# =========================================================

from pyspark.sql.window import Window


# =========================================================
# MACHINE LEARNING PIPELINE
# =========================================================

from pyspark.ml import Pipeline


# =========================================================
# FEATURE ENGINEERING
# =========================================================

from pyspark.ml.feature import (

    # Feature Vector Builder
    VectorAssembler,

    # Label Encoding
    StringIndexer

)


# =========================================================
# CLASSIFICATION MODELS
# =========================================================

from pyspark.ml.classification import (

    # Decision Tree Classification
    DecisionTreeClassifier

)


# =========================================================
# ENTERPRISE FORECASTING / REGRESSION MODELS
# =========================================================

from pyspark.ml.regression import (

    # Gradient Boosted Tree Regressor
    GBTRegressor

)


# =========================================================
# MODEL EVALUATION
# =========================================================

from pyspark.ml.evaluation import (

    # Classification Evaluation
    MulticlassClassificationEvaluator,

    # Regression Evaluation
    RegressionEvaluator

)

